# In-class Session: Train and Test ML/DL Models
AI for Materials Science — Hands-on session 5

Today we train and test machine-learning models that predict the **average voltage** of lithium
insertion electrodes from their composition. Sections A to D are worked examples: a linear baseline,
tree ensembles with cross-validation, and a neural network in PyTorch. In section E you train one
model yourself; that is what you submit for attendance.

**Do this first:** 1. File ▸ Save a copy in Drive. 2. In your copy, run 0-1 and 0-2 right away; they
install while we review. If you ran 0-1 before saving, run 0-1 and 0-2 again in your copy now.

## Today's plan

### 0 · Setup
Install two libraries, download the course data, import everything.

### A · The data and its features
Load the electrode table and turn each formula into the 145 composition features from session 2.

### B · Split, a linear baseline, three metrics
Split by composition, fit a linear model, read MAE, RMSE and R², and look at a parity plot.

### C · Random forest, gradient boosting, cross-validation
Compare three models with 5-fold cross-validation and see which features matter.

### D · A neural network in PyTorch
Build an MLP, watch the train and validation loss, and compare it with the trees.

### E · Assignment (attendance)
Pick one model (or the other target), train and evaluate it, and save a results table and a parity plot.

### F · The competition (optional)
A one-week Kaggle-style challenge on liquid-electrolyte data, with a starter notebook.

---

Data: Materials Project insertion electrodes (Li), CC BY 4.0 (modified: filtered). Jain et al., APL Mater. 1, 011002 (2013); accessed 2026-10-22.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

**Shared or lab PC?** Sign out of Google when you finish and choose Runtime ▸ Disconnect and delete runtime.

## 0. Setup

### 0-1. Install the libraries
Only `matminer` and `pymatgen` need installing; scikit-learn, PyTorch, pandas and Matplotlib ship with Colab.
The versions are pinned to the ones this notebook was tested with. The install takes a minute or two.

In [ ]:
!pip install -q matminer==0.10.1 pymatgen==2026.9.24

### 0-2. Download the course material

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries
The seeds make the random parts (splits, forests, network initialisation) repeatable.

In [ ]:
import copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:   ## only the live featurizer in A-3 needs these two libraries
    from pymatgen.core import Composition
    from matminer.featurizers.base import MultipleFeaturizer
    from matminer.featurizers.composition import (ElementProperty, IonProperty,
                                                  Stoichiometry, ValenceOrbital)
    HAVE_MATMINER = True
except ImportError as err:   ## 0-1 failed: A-3 reads the saved features instead
    HAVE_MATMINER = False
    print("matminer/pymatgen not importable; A-3 uses the saved features.", err)

np.random.seed(0)
torch.manual_seed(0)

### 0-4. Set the file paths

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data and its features

### A-1. Load the electrode table
One row is one lithium insertion electrode computed by the Materials Project: the host framework, the
formula at full discharge, and the average voltage against Li/Li⁺ along the lithiation path.

In [ ]:
df = pd.read_csv(DATA_DIR / "mp_li_electrodes.csv")
print("rows, columns:", df.shape)
cols = ["battery_formula", "formula_discharge",
        "average_voltage", "capacity_grav"]
df[cols].head()

### A-2. What one row means
`formula_discharge` is the composition with lithium in; `average_voltage` (V vs Li/Li⁺) is today's target;
`capacity_grav` (mAh/g) is the alternative target for the assignment. The voltage comes from DFT reaction energies
against Li metal; the capacity from how much Li the host takes up per gram. Both are computed, not measured.

In [ ]:
df[["average_voltage", "capacity_grav", "nelements"]].describe().round(2)

### A-3. From formula to 145 composition features
The same featurizer as in session 2: stoichiometry (6), Magpie element-property statistics (132),
valence-orbital fractions (4) and ionic attributes (3). If matminer could not be installed, or the computation
fails in Colab, the cell reads a saved copy of exactly the same features (computed by the TA with the same
pinned versions), so everyone continues with the same matrix.

In [ ]:
X_all, source = None, "saved copy (matminer not installed)"
if HAVE_MATMINER:
    try:   ## construction and computation are both optional
        featurizer = MultipleFeaturizer([
            Stoichiometry(), ElementProperty.from_preset("magpie"),
            ValenceOrbital(props=["frac"]), IonProperty(fast=True)])
        featurizer.set_n_jobs(1)
        comps = [Composition(f) for f in df["formula_discharge"]]
        feats = featurizer.featurize_many(comps, ignore_errors=True,
                                          pbar=False)
        X_all = pd.DataFrame(feats, index=df.index,
                             columns=featurizer.feature_labels())
        source = "computed now"
    except Exception as err:   ## either failed: use the saved copy below
        source = f"saved copy ({type(err).__name__})"
if X_all is None:   ## the saved copy of exactly the same 145 features
    saved = pd.read_csv(DATA_DIR / "mp_li_electrodes_magpie145.csv.gz")
    X_all = df[["battery_id"]].merge(saved, on="battery_id", how="left")
    X_all = X_all.drop(columns="battery_id")
FEATURE_NAMES = list(X_all.columns)
print("feature matrix:", X_all.shape, "from", source)

## B. Split, a linear baseline, three metrics

### B-1. Repeated compositions
The same discharged formula can appear in several rows (different host structures). If such rows land
in both train and test, the model is partly tested on data it has already seen.

In [ ]:
repeated = int(df["formula_discharge"].duplicated(keep=False).sum())
unique = df["formula_discharge"].nunique()
print(f"{repeated} rows share their formula with another row",
      f"({unique} unique formulas in {len(df)} rows)")

### B-2. Split by composition
`GroupShuffleSplit` keeps every row of one formula on the same side. 80% of the formulas train, 20% test.

In [ ]:
groups = df["formula_discharge"]
y_all = df["average_voltage"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=groups))
X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
groups_train = groups.iloc[train_idx]
leaked = set(groups_train) & set(groups.iloc[test_idx])
assert not leaked, "a formula leaked into both sets"
print("train rows:", len(X_train), " test rows:", len(X_test))

### B-3. A linear baseline
A pipeline fills missing feature values with the median, standardises every column, and fits ordinary
least squares. The imputer and scaler are fitted on the training rows only.

In [ ]:
linear = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LinearRegression())
linear.fit(X_train, y_train)

### B-4. Three metrics
MAE and RMSE are in volts; R² is dimensionless. Read the train and test columns side by side.

In [ ]:
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

def scores(model, X, y):
    return metrics(y, model.predict(X))

pd.DataFrame({"train": scores(linear, X_train, y_train),
              "test": scores(linear, X_test, y_test)}).round(3)

### B-5. Parity plot
True voltage on the x-axis, predicted on the y-axis, for the train and test rows.

In [ ]:
def parity(ax, y_true, y_pred, title, target="average voltage", unit="V"):
    ax.scatter(y_true, y_pred, s=10, alpha=0.6)
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_xlabel(f"true {target} ({unit})")
    ax.set_ylabel(f"predicted ({unit})")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_train, linear.predict(X_train), "Linear - train")
parity(axes[1], y_test, linear.predict(X_test), "Linear - test")
fig.tight_layout()
fig.savefig(OUTPUT / "B5_parity_linear.png", dpi=150, bbox_inches="tight")
plt.show()

## C. Random forest, gradient boosting, cross-validation

### C-1. Three models
The linear model again, a random forest of 200 trees, and histogram gradient boosting (which handles
missing values itself, so it needs no imputer).

In [ ]:
models = {
    "Linear": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LinearRegression()),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"),
                                  RandomForestRegressor(n_estimators=200,
                                                        n_jobs=-1, random_state=0)),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
}
scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

def cv_scores(model, cv):
    res = cross_validate(model, X_train, y_train, groups=groups_train,
                         cv=cv, scoring=scoring)
    return {"CV MAE": -res["test_MAE"].mean(), "CV RMSE": -res["test_RMSE"].mean(),
            "CV RMSE std": res["test_RMSE"].std(), "CV R2": res["test_R2"].mean()}

### C-2. Five-fold cross-validation on the training set
`GroupKFold` keeps formulas together inside each fold too. Each model is fitted five times; we report the
mean and spread of the validation RMSE. The test set is not touched.

In [ ]:
cv = GroupKFold(n_splits=5)
rows = [{"model": name, **cv_scores(model, cv)}
        for name, model in models.items()]
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

### C-3. Fit on train, score on test
Now each model is fitted once on all training rows and scored on the held-out test rows.

In [ ]:
fitted, test_rows = {}, []
for name, model in models.items():
    t0 = time.time()
    fitted[name] = model.fit(X_train, y_train)
    test_rows.append({"model": name, **scores(fitted[name], X_test, y_test),
                      "fit s": time.time() - t0})
test_table = pd.DataFrame(test_rows).set_index("model").round(3)
test_table

### C-4. Which features matter?
A random forest records how much each feature reduced the error across its trees. The top 15 are shown.

In [ ]:
rf = fitted["RandomForest"].named_steps["randomforestregressor"]
importance = pd.Series(rf.feature_importances_, index=FEATURE_NAMES)
importance = importance.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
importance.head(15)[::-1].plot.barh(ax=ax)
ax.set_xlabel("importance"); ax.set_title("Random forest - top 15 features")
fig.tight_layout()
fig.savefig(OUTPUT / "C4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### C-5. Practice 1
Change **one** hyperparameter of the random forest (for example `n_estimators`, `max_depth` or
`max_features`) and run the 5-fold cross-validation again. Report the CV RMSE before and after, and write
one sentence on what changed and why.

Write it yourself in the empty cell below. Reuse `cv`, `groups_train` and `cv_scores` from C-1 and C-2.

In [ ]:
## Write your code here.
## Order: copy the RandomForest pipeline with one changed parameter -> cv_scores(...) -> print both CV RMSEs -> one sentence

## D. A neural network in PyTorch

### D-1. A validation split
We hold out 20% of the **training** formulas as a validation set: it tells us how the network is doing
after every epoch. No decision about the network uses the test rows; the MLP's test score appears only in
D-7. The split comes first, so that the scaling in D-2 is fitted on the fit rows only; the validation rows
must not influence it.

In [ ]:
val_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
fit_idx, val_idx = next(val_split.split(X_train, y_train, groups=groups_train))
X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
y_fit = y_train.iloc[fit_idx].to_numpy(dtype=np.float32)
y_val = y_train.iloc[val_idx].to_numpy(dtype=np.float32)
print("fit rows:", len(X_fit), " validation rows:", len(X_val))

### D-2. Scale the inputs and build data loaders
Neural networks train badly on features with very different ranges. We impute and standardise with
statistics from the fit rows only, convert everything to float32 tensors, and let a `DataLoader` serve
mini-batches of 64 rows.

In [ ]:
prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
prep.fit(X_fit)

def to_tensor(a):
    return torch.tensor(np.asarray(a, dtype=np.float32))

X_fit_t = to_tensor(prep.transform(X_fit))
y_fit_t = to_tensor(y_fit).reshape(-1, 1)
X_val_t = to_tensor(prep.transform(X_val))
y_val_t = to_tensor(y_val).reshape(-1, 1)
X_test_s = prep.transform(X_test)
y_test_a = y_test.to_numpy(dtype=np.float32)
train_loader = DataLoader(TensorDataset(X_fit_t, y_fit_t),
                          batch_size=64, shuffle=True)
print("scaled fit matrix:", tuple(X_fit_t.shape),
      " batches per epoch:", len(train_loader))

### D-3. Build an MLP
145 inputs → 128 → 64 → 1 output, with ReLU between the layers. Every arrow is a trainable weight.

In [ ]:
def make_mlp(n_in, dropout=0.0):
    return nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

torch.manual_seed(0)
mlp = make_mlp(X_fit_t.shape[1])
n_params = sum(p.numel() for p in mlp.parameters())
print(mlp)
print("trainable parameters:", n_params)

### D-4. The training loop
The same four steps as in the preclass loop, now over mini-batches. After every epoch both losses are
measured with the same weights in evaluation mode, on the fit rows and on the validation rows, so the two
curves are comparable. We remember the weights of the epoch with the lowest validation loss and restore
them at the end (a validation checkpoint); the loop itself runs for 200 epochs or 60 seconds, whichever
comes first. First the function, then the call.

In [ ]:
def train_mlp(model, loader, X_val, y_val, epochs=200, lr=1e-3,
              weight_decay=0.0, max_seconds=60):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    X_fit_all, y_fit_all = loader.dataset.tensors   ## the whole fit set, for the epoch-end loss
    history = {"train": [], "val": []}
    best = {"loss": float("inf"), "epoch": 0, "state": None}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()              ## 1. forget the previous gradients
            loss = loss_fn(model(xb), yb)      ## 2. how wrong are we?
            loss.backward()                    ## 3. gradients
            optimizer.step()                   ## 4. one step downhill
        model.eval()                           ## same weights, same mode for both losses
        with torch.no_grad():
            train_loss = loss_fn(model(X_fit_all), y_fit_all).item()
            val_loss = loss_fn(model(X_val), y_val).item()
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if val_loss < best["loss"]:
            best = {"loss": val_loss, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
        if time.time() - start > max_seconds:
            break
    model.load_state_dict(best["state"])       ## restore the best validation checkpoint
    history.update(epochs_completed=epoch, seconds=time.time() - start,
                   best_epoch=best["epoch"])
    return history

In [ ]:
EPOCHS, MAX_SECONDS = 200, 60
history = train_mlp(mlp, train_loader, X_val_t, y_val_t,
                    epochs=EPOCHS, max_seconds=MAX_SECONDS)
epochs_completed = history["epochs_completed"]
seconds, best_epoch = history["seconds"], history["best_epoch"]
print(f"epochs completed {epochs_completed}/{EPOCHS}",
      f"in {seconds:.1f} s; best epoch {best_epoch}")

### D-5. Loss curves
Training loss keeps falling as long as you train; validation loss stops improving once the network starts
memorising the training rows, and may rise again. That turning point is where overfitting begins; the
dotted line marks the checkpoint we kept.

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(h["train"], label="train"); ax.plot(h["val"], label="validation")
    ax.axvline(h["best_epoch"] - 1, color="gray", linestyle=":",
               label=f"best epoch {h['best_epoch']}")
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss (V²)")
    ax.set_title(title); ax.legend()
    return fig

fig = plot_history(history, "MLP - train vs validation loss")
fig.savefig(OUTPUT / "D5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### D-6. Practice 2
Add **one** regulariser to the MLP: dropout (`make_mlp(n_in, dropout=0.2)`), weight decay
(`weight_decay=1e-4` in `train_mlp`), or a smaller training budget (fewer `epochs`). Train it with the same
loop, plot its curves with `plot_history`, and compare its best validation RMSE with the unregularised
network's. No test-based tuning: regularisers are chosen on validation data, and the MLP's test score
appears only in D-7.

Write it yourself in the empty cell below.

In [ ]:
## Write your code here.
## Order: torch.manual_seed(0) -> make_mlp(...) -> train_mlp(...) -> plot_history(...) -> best validation RMSE for both networks

### D-7. ML or DL? Four models on one test set
The three scikit-learn models from C-3 and the MLP, scored once on the same held-out rows. Every choice
so far was made with cross-validation or the validation set; this table is the final report, not a place
to pick a winner and tune again. Read the error columns and the fitting time together.

In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(to_tensor(X_test_s)).numpy().ravel()
four = test_table.copy()
four.loc["MLP (PyTorch)"] = {**metrics(y_test_a, mlp_pred),
                             "fit s": history["seconds"]}
four = four.round(3)
four

In [ ]:
trees = ["RandomForest", "HistGradientBoosting"]
best_tree = cv_table.loc[trees, "CV RMSE"].idxmin()   ## chosen by CV in C-2, not by test
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_test, fitted[best_tree].predict(X_test), f"{best_tree} - test")
parity(axes[1], y_test, mlp_pred, "MLP - test")
fig.tight_layout()
fig.savefig(OUTPUT / "D7_parity_tree_vs_mlp.png", dpi=150, bbox_inches="tight")
plt.show()

## E. Assignment (attendance)

Sections A to D were worked examples. Now you run the same procedure yourself: choose one of the four
models, or keep a model and switch the target to gravimetric capacity, then train it, evaluate it on the
test rows, and save a results table and a parity plot. The two files you save are what you submit on KLMS.

### E-1. Set your student ID
Your ID is only used to name the files you submit.

In [ ]:
## Replace this with your own student ID; it only names your files.
STUDENT_ID = "demo"

if STUDENT_ID == "demo":
    raise ValueError("Set STUDENT_ID to your own ID before running this cell.")

print("Student ID:", STUDENT_ID)

### E-2. Choose your model and target
`MY_MODEL` is one of `linear`, `random_forest`, `hist_gradient_boosting`, `mlp`. `MY_TARGET` is
`average_voltage` (today's target) or `capacity_grav` (mAh/g). Change at least one of them from the defaults.

In [ ]:
## MY_MODEL: linear | random_forest | hist_gradient_boosting | mlp
MY_MODEL = "random_forest"
## MY_TARGET: average_voltage | capacity_grav
MY_TARGET = "average_voltage"

MODELS = {"linear", "random_forest", "hist_gradient_boosting", "mlp"}
assert MY_MODEL in MODELS, "unknown MY_MODEL"
assert MY_TARGET in {"average_voltage", "capacity_grav"}, "bad MY_TARGET"
y_my = df[MY_TARGET]
y_my_train, y_my_test = y_my.iloc[train_idx], y_my.iloc[test_idx]
print("model:", MY_MODEL, " target:", MY_TARGET,
      " train rows:", len(y_my_train), " test rows:", len(y_my_test))

### E-3. Train, evaluate and plot
Fit your model on `X_train` / `y_my_train` and evaluate it on both splits. Produce two objects:

1. `my_results` — a DataFrame with exactly these columns: `student_id, model, target, split, MAE, RMSE, R2`,
   one row for `train` and one for `test`.
2. `my_fig` — a Matplotlib figure with the test-set parity plot. Reuse `parity` and pass
   `target=MY_TARGET` with `unit="V"` for the voltage or `unit="mAh/g"` for the capacity.

Write it yourself in the empty cell below. For `mlp`, follow D-1 and D-2 (split first, then fit the scaler
on the fit rows) and reuse `to_tensor`, `make_mlp` and `train_mlp`.

In [ ]:
## Write your code here.
## Order: build the model for MY_MODEL -> fit on X_train, y_my_train -> scores for train and test -> my_results -> my_fig

### E-4. Save your submission
Running this cell writes two files and, in Colab, downloads them to your computer (allow the two
downloads if the browser asks). Upload exactly these two files on KLMS (W10 Lab); nothing is filled in by
hand. If E-3 is not finished yet, the cell stops with "Complete E-3 first" and no files are written.

1. `hos5_<id>_results.csv` — your results table
2. `hos5_<id>_parity.png` — your test-set parity plot

In [ ]:
## Stop with a clear message if E-3 has not been completed.
missing = [n for n in ("my_results", "my_fig") if n not in globals()]
if missing:
    raise NameError(f"Complete E-3 first. Not defined yet: {missing}")
columns = ["student_id", "model", "target", "split", "MAE", "RMSE", "R2"]
assert list(my_results.columns) == columns, f"columns must be {columns}"
safe_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-")
stem = OUTPUT / f"hos5_{safe_id}"
paths = [Path(f"{stem}_results.csv"), Path(f"{stem}_parity.png")]
my_results.to_csv(paths[0], index=False)
my_fig.savefig(paths[1], dpi=180, bbox_inches="tight")
for path in paths:
    print("created" if path.stat().st_size > 0 else "MISSING", path)
print(my_results.round(3).to_string(index=False))
try:   ## in Colab, download both files to your computer for the KLMS upload
    from google.colab import files
    files.download(str(paths[0])); files.download(str(paths[1]))
except ImportError:
    print("Not in Colab: download the two files from the outputs folder.")
print("Submit both files to KLMS (W10 Lab).")

## F. The competition (optional)

From today 17:30 until Sunday 23:59 KST you can take part in a Kaggle-style challenge: predict the ionic
conductivity of liquid electrolytes (CALiSol-23) and climb a public leaderboard. The site link is at the
top of the **advanced** notebook, which is also the starter you can submit from after one run.

- Task: predict ionic conductivity k (mS/cm) of liquid electrolytes, CALiSol-23 test split. Metric: RMSE on the hidden labels (lower is better).
- Public leaderboard: 30% of the test formulations; private: the remaining 70%, revealed after the deadline.
- Open: 2026-11-02 17:30 KST. Deadline: 2026-11-08 23:59 KST. Private results: 2026-11-09 09:00 KST.
- Submission: a CSV with the header id,k covering every test id, plus a Colab or Google Drive link to your notebook.
- Limit: 5 accepted submissions per student per KST day. Rejected files do not count.
- Identity: your student ID and the 6-digit access code sent to you on KLMS. Only your nickname is shown.
- Final ranking uses the private RMSE of each student's best public submission.
- Do not look up or reuse the original CALiSol-23 values for the test formulations; build a model from the training file. Suspicious entries are flagged and may be disqualified.
- The top 3 notebooks are reproduced by the TA before results are announced in class on Nov 9.
- This is an optional activity; any bonus credit is decided by the instructor.
- Too many failed logins trigger a 15-minute cool-down; 30 failures in a day lock that ID for the day.
- Locked? Contact the TA or the site owner; a Space restart clears locks.

## Data sources

### Materials Project insertion electrodes
`Data/mp_li_electrodes.csv` and `Data/mp_li_electrodes_magpie145.csv.gz`. Materials Project insertion electrodes (working ion Li), accessed 2026-10-22, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Cite: A. Jain, S. P. Ong, G. Hautier, W. Chen, W. D. Richards, S. Dacek, S. Cholia, D. Gunter, D. Skinner, G. Ceder, K. A. Persson, *APL Materials* 1, 011002 (2013), https://doi.org/10.1063/1.4812323. Changes: rows without average_voltage or formula_discharge removed; 0 < V < 6 kept; duplicate battery_id removed; composition features added with matminer (Ward et al., *Comput. Mater. Sci.* 152, 60 (2018), https://doi.org/10.1016/j.commatsci.2018.05.018) and Magpie properties (Ward et al., *npj Comput. Mater.* 2, 16028 (2016), https://doi.org/10.1038/npjcompumats.2016.28).